### Homework 3

In [1]:
!curl -o homework3.csv https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  272k  100  272k    0     0  2538k      0 --:--:-- --:--:-- --:--:-- 2544k


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [4]:
df = pd.read_csv("homework3.csv")
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [5]:
df.head().T

,0,1,2,3,4
lead_source,organic_search,social_media,referral,paid_ads,referral
industry,technology,technology,retail,manufacturing,manufacturing
employment_status,employed,employed,employed,student,student
location,europe,south_america,europe,NaN,north_america
annual_income,88160.0,72688.0,44697.0,NaN,24062.0
number_of_courses_viewed,3,3,3,3,4
interaction_count,4,7,4,6,5
lead_score,0.64,0.72,0.58,0.57,0.62
converted,1,1,1,0,1


In [6]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [15]:
numerical = df.columns[df.dtypes != 'string'].tolist()
numerical

['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score',
 'converted']

In [16]:
categorical = df.columns[df.dtypes == 'string'].tolist()
categorical

['lead_source', 'industry', 'employment_status', 'location']

In [17]:
for col in df.columns:
    if col in categorical:
        df[col] = df[col].fillna('NA')
    else:
        df[col] = df[col].fillna(0.0)


In [18]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

### Q1

In [ ]:
df.industry.value_counts(normalize=True)


industry
technology       0.246429
retail           0.194328
healthcare       0.176471
finance          0.151261
education        0.134244
manufacturing    0.097269
Name: proportion, dtype: float64

### Q2

In [19]:
df[numerical].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


### Q3

In [34]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [35]:
y_train = df_train.pop('converted').values
y_val = df_val.pop('converted').values
df_test = df_test.drop(columns='converted')

In [36]:
df_train.columns

Index(['lead_source', 'industry', 'employment_status', 'location',
       'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score'],
      dtype='str')

In [37]:
from sklearn.metrics import mutual_info_score, accuracy_score

In [38]:
{c: round(mutual_info_score(y_train, df_train[c]), 2) for c in categorical}

{'lead_source': 0.03,
 'industry': 0.0,
 'employment_status': 0.02,
 'location': 0.0}

In [39]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

### Q4

In [41]:
def train_and_score(features, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict('records'))
    X_val = dv.transform(df_val[features].to_dict('records'))
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_val, model.predict(X_val))

features = df_train.columns.tolist()
print(features)
base_acc = train_and_score(features)
round(base_acc, 2)

['lead_source', 'industry', 'employment_status', 'location', 'annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


0.65

### Q5

In [44]:
exclude_feature = ['lead_source','number_of_courses_viewed','interaction_count']

for feature in exclude_feature:
    features = [f for f in df_train.columns if f != feature]
    acc = train_and_score(features)
    print(f"Accuracy without {feature}: {round(acc, 2)}")
    print(f"Difference to Base Accuracy {feature}: {round(base_acc - acc, 4)}")


Accuracy without lead_source: 0.64
Difference to Base Accuracy lead_source: 0.003
Accuracy without number_of_courses_viewed: 0.64
Difference to Base Accuracy number_of_courses_viewed: 0.002
Accuracy without interaction_count: 0.6
Difference to Base Accuracy interaction_count: 0.044


### Q6

In [47]:
C = [0.000001, 0.00001, 0.0001, 0.001]
for c in C:
    features = df_train.columns.tolist()
    acc = train_and_score(features, C=c)
    print(f"Accuracy with C={c}: {round(acc, 2)}")


Accuracy with C=1e-06: 0.6
Accuracy with C=1e-05: 0.6
Accuracy with C=0.0001: 0.61
Accuracy with C=0.001: 0.65
